# 🚀 AI-Powered Customer Support Triage System
**Business Problem:** The support team spends hours manually reading and routing hundreds of messy customer tickets daily.
**Our Solution:** An automated NLP pipeline using Meta's Llama 3.3 (via Groq) to instantly categorize issues, detect sentiment, and summarize complaints, followed by an analytics dashboard.

In [2]:
import pandas as pd
import json
import requests
import time
import plotly.express as px

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [3]:
# Load the dataset
file_name = 'customer_support_tickets.csv'
df = pd.read_csv(file_name)

# Take a sample for the live demonstration
df_sample = df.head(10).copy()
print(f"✅ Loaded {len(df_sample)} tickets for processing.")

✅ Loaded 10 tickets for processing.


In [4]:
# Display just the raw text column
pd.set_option('display.max_colwidth', None)
display(df_sample[['Ticket ID', 'Ticket Description']].head(10))

,Ticket ID,Ticket Description
0,1,"I'm having an issue with the {product_purchased}. Please assist.\n\nYour billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists."
1,2,"I'm having an issue with the {product_purchased}. Please assist.\n\nIf you need to change an existing product.\n\nI'm having an issue with the {product_purchased}. Please assist.\n\nIf The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly."
2,3,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.\n\n1.8.3 I really I'm using the original charger that came with my {product_purchased}, but it's not charging properly."
3,4,"I'm having an issue with the {product_purchased}. Please assist.\n\nIf you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved."
4,5,I'm having an issue with the {product_purchased}. Please assist.\n\n\nNote: The seller is not responsible for any damages arising out of the delivery of the battleground game. Please have the game in good condition and shipped to you I've noticed a sudden decrease in battery life on my {product_purchased}. It used to last much longer.
5,6,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond. To remove the new {product_purch I've checked for any available software updates for my {product_purchased}, but there are none."
6,7,"I'm unable to access my {product_purchased} account. It keeps displaying an 'Invalid Credentials' error, even though I'm using the correct login information. How can I regain access to my account?\n\nSolution 1 I'm unable to find the option to perform the desired action in the {product_purchased}. Could you please guide me through the steps?"
7,8,"I'm having an issue with the {product_purchased}. Please assist. (Thanks) I will contact all my suppliers and confirm.\n\nPlease try and find out whether their inventory is currently stocked, or any other reason. I am I've performed a factory reset on my {product_purchased}, hoping it would resolve the problem, but it didn't help."
8,9,"I'm having an issue with the {product_purchased}. Please assist. Thank you.\n\n{product_purchased} is not the exact type you might prefer, they use the exact same method for different uses. Please help I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?"
9,10,"My {product_purchased} is making strange noises and not functioning properly. I suspect there might be a hardware issue. Can you please help me with this?\n\n} If we can, please send a ""request"" to dav The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly."


In [5]:
# Insert your Groq API Key here
API_TOKEN = "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"
API_URL = "https://api.groq.com/openai/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}
print("✅ API Configuration Set.")

✅ API Configuration Set.


In [6]:
def build_prompt(ticket_text):
    """Creates a strict set of rules for the AI to follow."""
    return f"""
    Analyze this customer support ticket.
    Categories allowed: Bug, Billing, Feature Request, How-To, Praise, Hardware Issue.
    Sentiments allowed: Positive, Neutral, Negative.

    Ticket: "{ticket_text}"

    Required JSON output format:
    {{
        "Category": "...",
        "Sentiment": "...",
        "Summary": "A strict 10-word summary of the issue."
    }}
    """

In [7]:
def analyze_ticket_with_ai(ticket_text):
    """Sends the prompt to Groq and returns a parsed dictionary."""
    prompt = build_prompt(ticket_text)

    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {"role": "system", "content": "You are a data extraction tool. You always output perfectly formatted JSON."},
            {"role": "user", "content": prompt}
        ],
        "response_format": {"type": "json_object"},
        "temperature": 0.1
    }

    try:
        response = requests.post(API_URL, headers=HEADERS, json=payload)
        result_text = response.json()['choices'][0]['message']['content']
        return json.loads(result_text)
    except Exception as e:
        return {"Category": "Error", "Sentiment": "Error", "Summary": "Failed."}

In [8]:
# Testing a tricky phrase
test_ticket = "I love the new design, but I was charged twice this month and I'm furious!"
print("Testing AI Engine...")

result = analyze_ticket_with_ai(test_ticket)
print(json.dumps(result, indent=4))

Testing AI Engine...
{
    "Category": "Billing",
    "Sentiment": "Negative",
    "Summary": "Customer charged twice and extremely unhappy about it"
}


In [9]:
# Create empty lists to store our AI-generated data
categories = []
sentiments = []
summaries = []

print("✅ Data structures initialized and ready for loop.")

✅ Data structures initialized and ready for loop.


In [10]:
print("🚀 Starting AI processing loop...\n")

for index, row in df_sample.iterrows():
    text = str(row['Ticket Description'])
    print(f"Processing Ticket ID {row['Ticket ID']}...")

    # Process via AI
    ai_result = analyze_ticket_with_ai(text)

    # Store results
    categories.append(ai_result.get('Category'))
    sentiments.append(ai_result.get('Sentiment'))
    summaries.append(ai_result.get('Summary'))

    time.sleep(1) # Be polite to the API rate limits

print("\n✅ Pipeline complete!")

🚀 Starting AI processing loop...

Processing Ticket ID 1...
Processing Ticket ID 2...
Processing Ticket ID 3...
Processing Ticket ID 4...
Processing Ticket ID 5...
Processing Ticket ID 6...
Processing Ticket ID 7...
Processing Ticket ID 8...
Processing Ticket ID 9...
Processing Ticket ID 10...

✅ Pipeline complete!


In [11]:
# Add AI data back to the dataframe
df_sample['AI_Category'] = categories
df_sample['AI_Sentiment'] = sentiments
df_sample['AI_Summary'] = summaries

# Save to a new CSV
df_sample.to_csv("enriched_tickets.csv", index=False)
print("✅ Data successfully saved to 'enriched_tickets.csv'")

✅ Data successfully saved to 'enriched_tickets.csv'


In [12]:
# Display the final, clean table
display(df_sample[['Ticket ID', 'AI_Category', 'AI_Sentiment', 'AI_Summary']])

,Ticket ID,AI_Category,AI_Sentiment,AI_Summary
0,1,Bug,Neutral,Customer needs help with product issue persists still
1,2,Bug,Neutral,Customer experiencing intermittent issue with product purchased
2,3,Hardware Issue,Negative,Product not turning on or charging properly suddenly
3,4,Bug,Negative,Unresolved issue with product after multiple support requests
4,5,Hardware Issue,Negative,Battery life decreased on purchased product suddenly
5,6,Hardware Issue,Negative,Product not turning on after working fine yesterday suddenly
6,7,Bug,Negative,User cannot access account due to credentials issue
7,8,Bug,Neutral,Customer needs help with faulty product issue resolution
8,9,Bug,Neutral,Firmware update caused issue with product functionality suddenly
9,10,Hardware Issue,Negative,Product making strange noises and not functioning properly sometimes
